# 콜비 어투 QLoRA 파인튜닝 (Qwen2.5-7B + Unsloth)

청약콜 C팀 2차 — 공모주 안내 페르소나 챗봇 **콜비**의 어투를 로컬 오픈모델에 학습시킨다.

- **모델**: `unsloth/Qwen2.5-7B-Instruct-bnb-4bit` (무거우면 3B로 교체 — 셀 참고)
- **데이터**: `colbi_sft_train.jsonl`(180) / `colbi_sft_val.jsonl`(20) — chat 포맷, system 없이 user/assistant 2턴
- **방식**: QLoRA(4bit) + LoRA 어댑터, assistant 응답만 학습(train_on_responses_only)
- **산출**: LoRA 어댑터 → GGUF(q4_k_m) → Ollama `colbi-qwen`

> ⚠️ 런타임 → 런타임 유형 변경 → **GPU(T4)** 로 설정하고 실행하세요. 무료 T4(16GB)에서 7B 4bit QLoRA 동작합니다.

## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. Unsloth 설치

In [ ]:
%%capture
!pip install unsloth
# 최신 버전 강제(선택): 문제 시 아래 주석 해제
# !pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

## 2. 모델 로드 (Qwen2.5-7B, 4bit)

OOM(메모리 부족) 나면 `model_name`을 `unsloth/Qwen2.5-3B-Instruct-bnb-4bit`로 바꾸세요. (데이터·이후 셀 그대로 재사용)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048   # 콜비 답변은 짧아서 충분

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,          # 자동(T4=fp16)
    load_in_4bit = True,
)

## 3. LoRA 어댑터 부착

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 4. 데이터 업로드 & 포맷

로컬 `_gitcheck/data/`의 **`colbi_sft_train.jsonl`, `colbi_sft_val.jsonl`** 두 파일을 업로드하세요.
(train/val은 gitignore라 레포엔 없음 → `python -m backend.scripts.prep_sft_data`로 생성한 그 파일)

In [ ]:
# ── 옵션 A: 파일 직접 업로드 (권장) ──
from google.colab import files
print("colbi_sft_train.jsonl 과 colbi_sft_val.jsonl 을 선택하세요")
uploaded = files.upload()

# ── 옵션 B: 레포 클론 후 전처리로 재생성 (레포 public일 때) ──
# !git clone https://github.com/j43hyun9/-cheongyak-call-.git repo
# %cd repo
# !git checkout develop && python -m backend.scripts.prep_sft_data
# %cd ..

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

# Qwen2.5 = ChatML 템플릿
tokenizer = get_chat_template(tokenizer, chat_template = "qwen-2.5")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
             for c in convos]
    return {"text": texts}

train_ds = load_dataset("json", data_files="colbi_sft_train.jsonl", split="train")
val_ds   = load_dataset("json", data_files="colbi_sft_val.jsonl",   split="train")
train_ds = train_ds.map(formatting_prompts_func, batched=True)
val_ds   = val_ds.map(formatting_prompts_func,   batched=True)

print("train:", len(train_ds), "| val:", len(val_ds))
print("\n--- 샘플 (템플릿 적용 후) ---\n")
print(train_ds[0]["text"])

## 5. 학습 설정 & 실행

`train_on_responses_only` → user 질문은 loss에서 빼고 **assistant 답변(콜비 어투)만 학습**한다.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,          # 어투 학습(style transfer)엔 2~3 적당
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

# assistant 응답만 학습 (Qwen ChatML 마커 기준)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

In [ ]:
trainer_stats = trainer.train()
trainer_stats

## 6. 추론 테스트 — 어투 확인

카테고리별로 콜비 어투·가드레일·할루시 방지가 살아있는지 눈으로 확인.

In [ ]:
FastLanguageModel.for_inference(model)   # 추론 모드(2x 빠름)

def ask(q, max_new_tokens=256):
    msgs = [{"role": "user", "content": q}]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens,
                         temperature=0.7, top_p=0.9, do_sample=True)
    text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Q: {q}\nA: {text}\n" + "-"*60)

ask("공모주가 뭐야?")                    # 개념
ask("공모주 청약 어떻게 신청해?")          # 절차
ask("이번 주에 청약하는 공모주 있어?")      # 일정 → 캘린더 유도(지어내면 안 됨)
ask("이 종목 사도 돼? 오를까?")            # 금지행동 → 거절
ask("테슬라 공모주 언제야?")               # 할루시 → 목록에 없음
ask("어른처럼 진지하게 말해봐")            # 페르소나 유지

## 7. 저장 — LoRA 어댑터 & GGUF(Ollama용)

In [ ]:
# (1) LoRA 어댑터만 저장 (재학습·백업용)
model.save_pretrained("colbi_lora")
tokenizer.save_pretrained("colbi_lora")

# (2) 병합 + GGUF 양자화 내보내기 (Ollama가 읽는 포맷)
model.save_pretrained_gguf("colbi_gguf", tokenizer, quantization_method="q4_k_m")
# → colbi_gguf/ 안에 *.Q4_K_M.gguf 생성

In [ ]:
# GGUF 파일명 확인 후 Ollama Modelfile 작성
import glob, os
gguf = glob.glob("colbi_gguf/*.gguf")[0]
print("GGUF:", gguf)

modelfile = f"""FROM {os.path.basename(gguf)}
PARAMETER temperature 0.7
PARAMETER top_p 0.9
"""
with open("colbi_gguf/Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile)
print(modelfile)

In [ ]:
# GGUF + Modelfile 다운로드 (로컬 Ollama로 옮기기)
from google.colab import files
files.download(gguf)
files.download("colbi_gguf/Modelfile")

## 7-2. Hugging Face Hub 업로드 (팀 공유용)

모델 가중치는 git에 넣지 않고 **HF Hub**에 올려 팀이 받아쓴다. (파일 직접 전달이 편하면 이 셀은 건너뛰고 위 7번 다운로드본을 공유해도 됨)

- HF 토큰 발급: huggingface.co/settings/tokens → **write** 권한
- `HF_ID`를 본인 HF 계정으로 바꿀 것
- 업로드 후 임강님: `huggingface-cli download <HF_ID>/colbi-qwen-gguf --local-dir colbi_gguf` 로 받아 `ollama create`

In [ ]:
from huggingface_hub import login
login()   # 프롬프트에 HF write 토큰 입력 (또는 login(token="hf_xxx"))

HF_ID = "your-hf-id"          # ← 본인 HF 계정으로 변경
GGUF_REPO = f"{HF_ID}/colbi-qwen-gguf"
LORA_REPO = f"{HF_ID}/colbi-qwen-lora"

# (1) GGUF를 HF에 업로드 — 임강님이 여기서 받아 ollama create
model.push_to_hub_gguf(GGUF_REPO, tokenizer, quantization_method="q4_k_m")

# (2) (선택) LoRA 어댑터 백업 — 재학습/이어붙이기용
model.push_to_hub(LORA_REPO, tokenizer)

print(f"업로드 완료 → https://huggingface.co/{GGUF_REPO}")

## 8. 로컬에서 Ollama 등록 & 백엔드 연결

다운받은 `*.gguf` + `Modelfile`을 같은 폴더에 두고 (임강 담당):

```bash
# 1) Ollama에 콜비 모델 등록
ollama create colbi-qwen -f Modelfile

# 2) 테스트
ollama run colbi-qwen "공모주가 뭐야?"
```

백엔드 연결 — `.env` 두 줄만 바꾸면 전환됨 (PR#8 local 엔진):

```
LLM_ENGINE=local
OLLAMA_MODEL=colbi-qwen
# OLLAMA_BASE_URL=http://localhost:11434/v1  (기본값)
```

그 후 `uvicorn backend.main:app --reload` → `/chat`이 파인튜닝된 콜비로 응답.

---
### 다음: 파인튜닝 전/후 평가
`eval/evalset.csv`(30문항, held-out)로 **base Qwen vs 파인튜닝 콜비** 응답을 비교해 평가 리포트에 반영.
> ⚠️ 평가 전에 평가셋 누수 7문항 교체할 것(전/후 비교 신뢰도).